# Control 1 optimizacion NRC 7924 #
 Integrantes:

 -Vicente Díaz

 -Eduardo Zepeda

 -Fernando Chavez
  

In [ ]:
!pip install -q amplpy
from amplpy import AMPL, ampl_notebook

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


---
## Ejercicio 1 - La Química (minimización de costos)

**Variables:**
- `x1`: horas proceso 1 (costo $40/hr, produce 3A + 1B + 1C)

- `x2`: horas proceso 2 (costo $10/hr, produce 1A + 1B)

**Objetivo:** Minimizar z = 40x1 + 10x2

**Restricciones:**
- Demanda A: 3x1 + x2 >= 40
- Demanda B: x1 + x2 >= 15
- Demanda C: x1 >= 5

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var x1 >= 0;  # horas proceso 1
var x2 >= 0;  # horas proceso 2

# definicion de funcion objetivo

minimize costo: 40 * x1 + 10 * x2;

# definicion de restricciones

s.t. demanda_A: 3 * x1 + x2 >= 40;
s.t. demanda_B: x1 + x2 >= 15;
s.t. demanda_C: x1 >= 5;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados del plan de produccion:")

cant_x1 = ampl.get_variable('x1').value()
cant_x2 = ampl.get_variable('x2').value()
costo   = ampl.get_objective('costo').value()

print("horas proceso 1: ", cant_x1)
print("horas proceso 2: ", cant_x2)
print("costo minimo: ", costo)

# Analisis de sensibilidad
print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados del plan de produccion:
horas proceso 1:  5.0
horas proceso 2:  25.0
costo minimo:  450.0

Analisis de sensibilidad
demanda_A: precio sombra = 10.00, holgura = 0.00
demanda_B: precio sombra = 0.00, holgura = 15.00
demanda_C: precio sombra = 10.00, holgura = 0.00


---
## Ejercicio 2 - Charcha (Maximización de utilidades)

**Variables:**
- `m1`: libras materia prima para insumo 1
- `m2`: libras materia prima para insumo 2
- `xA, xB, xC, xD`: onzas productos A, B, C, D
- `r`: onzas desechadas al río (máx 1000)

**Objetivo:** Maximizar z = 17xA + 16xB + 7xC + 2xD - 8m1 - 10m2

(Utilidad = ingresos - materia prima - procesamiento - ejecución)

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var m1 >= 0;  # libras materia prima -> insumo 1
var m2 >= 0;  # libras materia prima -> insumo 2
var xA >= 0;  # onzas producto A
var xB >= 0;  # onzas producto B
var xC >= 0;  # onzas producto C
var xD >= 0;  # onzas producto D
var r  >= 0;  # onzas desechadas al rio

# definicion de funcion objetivo
# utilidad = ingresos - (mat prima + procesamiento + ejecucion)

maximize utilidad: 17 * xA + 16 * xB + 7 * xC + 2 * xD - 8 * m1 - 10 * m2;

# definicion de restricciones

s.t. balance_insumo1: 2 * xA + xB + 2 * xC <= 2 * m1;
s.t. balance_insumo2: xA + 2 * xB + 2 * xD <= 3 * m2;
s.t. balance_desecho: xA + 0.8 * xB = 0.8 * xC + 1.2 * xD + r;
s.t. limite_rio: r <= 1000;
s.t. tiempo_proceso: 2 * m1 + 4 * m2 + xA + 8 * xB + 4 * xC + 5 * xD <= 6000;
s.t. demanda_max_A: xA <= 5000;
s.t. demanda_max_B: xB <= 5000;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 6288.732394
5 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados de produccion semanal:")

cant_m1 = ampl.get_variable('m1').value()
cant_m2 = ampl.get_variable('m2').value()
cant_xA = ampl.get_variable('xA').value()
cant_xB = ampl.get_variable('xB').value()
cant_xC = ampl.get_variable('xC').value()
cant_xD = ampl.get_variable('xD').value()
cant_r = ampl.get_variable('r').value()
utilidad = ampl.get_objective('utilidad').value()

print("materia prima insumo 1: ", cant_m1)
print("materia prima insumo 2: ", cant_m2)
print("producto A (oz): ", cant_xA)
print("producto B (oz): ", cant_xB)
print("producto C (oz): ", cant_xC)
print("producto D (oz): ", cant_xD)
print("desecho al rio (oz): ", cant_r)
print("utilidad maxima: ", utilidad)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados de produccion semanal:
materia prima insumo 1:  1316.901408450705
materia prima insumo 2:  380.2816901408448
producto A (oz):  1140.845070422535
producto B (oz):  0.0
producto C (oz):  176.0563380281689
producto D (oz):  0.0
desecho al rio (oz):  1000.0
utilidad maxima:  6288.7323943661895

Analisis de sensibilidad
balance_insumo1: precio sombra = 4.37, holgura = 0.00
balance_insumo2: precio sombra = 3.83, holgura = -0.00
balance_desecho: precio sombra = 4.05, holgura = -0.00
limite_rio: precio sombra = 4.05, holgura = 0.00
tiempo_proceso: precio sombra = 0.37, holgura = 0.00
demanda_max_A: precio sombra = 0.00, holgura = 3859.15
demanda_max_B: precio sombra = 0.00, holgura = 5000.00


---
## Ejercicio 3 - Refinería de Melrose (Maximización)

**Variables:**
- `p1, p2, p3`: barriles procesados por método 1, 2, 3
- `c68`: barriles grado 6 enviados a craqueo → grado 8 (costo $1.30)
- `c810`: barriles grado 8 enviados a craqueo → grado 10 (costo $2.00)
- `g6, g8, g10`: barriles de cada grado destinados a gasolina
- `a6, a8, a10`: barriles de cada grado destinados a aceite
- `d6, d8, d10`: barriles desechados de cada grado (costo $0.20)

**Objetivo:** Maximizar z = ingresos - procesamiento - craqueo - desecho

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var p1  >= 0;  # barriles metodo 1
var p2  >= 0;  # barriles metodo 2
var p3  >= 0;  # barriles metodo 3
var c68  >= 0; # barriles grado 6
var c810 >= 0; # barriles grado 8
var g6  >= 0;  # grado 6 a gasolina
var g8  >= 0;  # grado 8 a gasolina
var g10 >= 0;  # grado 10 a gasolina
var a6  >= 0;  # grado 6 a aceite
var a8  >= 0;  # grado 8 a aceite
var a10 >= 0;  # grado 10 a aceite
var d6  >= 0;  # grado 6 desechado
var d8  >= 0;  # grado 8 desechado
var d10 >= 0;  # grado 10 desechado

# definicion de funcion objetivo
# ingresos - procesamiento - craqueo - desecho

maximize utilidad:
    8 * (g6 + g8 + g10) + 6 * (a6 + a8 + a10)
    - 3.4 * p1 - 3 * p2 - 2.6 * p3
    - 1.3 * c68 - 2 * c810
    - 0.2 * (d6 + d8 + d10);

# definicion de restricciones

s.t. balance_grado6: 0.2 * p1 + 0.3 * p2 + 0.4 * p3 = c68 + g6 + a6 + d6;
s.t. balance_grado8: 0.2 * p1 + 0.3 * p2 + 0.4 * p3 + c68 = c810 + g8 + a8 + d8;
s.t. balance_grado10: 0.6 * p1 + 0.4 * p2 + 0.2 * p3 + c810 = g10 + a10 + d10;
s.t. grado_gasolina: -3 * g6 - g8 + g10 >= 0;
s.t. grado_aceite: -a6 + a8 + 3 * a10 >= 0;
s.t. lim_gasolina: g6 + g8 + g10 <= 2000;
s.t. lim_aceite: a6 + a8 + a10 <= 600;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 11230
9 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados de la refineria:")

cant_p1  = ampl.get_variable('p1').value()
cant_p2  = ampl.get_variable('p2').value()
cant_p3  = ampl.get_variable('p3').value()
cant_c68 = ampl.get_variable('c68').value()
cant_c810 = ampl.get_variable('c810').value()
cant_g6  = ampl.get_variable('g6').value()
cant_g8  = ampl.get_variable('g8').value()
cant_g10 = ampl.get_variable('g10').value()
cant_a6  = ampl.get_variable('a6').value()
cant_a8  = ampl.get_variable('a8').value()
cant_a10 = ampl.get_variable('a10').value()
utilidad = ampl.get_objective('utilidad').value()

print("barriles metodo 1: ", cant_p1)
print("barriles metodo 2: ", cant_p2)
print("barriles metodo 3: ", cant_p3)
print("craqueo grado 6->8: ", cant_c68)
print("craqueo grado 8->10: ", cant_c810)
print("gasolina (total bbl): ", cant_g6 + cant_g8 + cant_g10)
print("aceite (total bbl): ", cant_a6 + cant_a8 + cant_a10)
print("utilidad maxima: ", utilidad)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados de la refineria:
barriles metodo 1:  0.0
barriles metodo 2:  2400.0
barriles metodo 3:  200.0
craqueo grado 6->8:  500.0
craqueo grado 8->10:  0.0
gasolina (total bbl):  2000.0
aceite (total bbl):  600.0
utilidad maxima:  11230.0

Analisis de sensibilidad
balance_grado6: precio sombra = -1.55, holgura = 0.00
balance_grado8: precio sombra = -2.85, holgura = 0.00
balance_grado10: precio sombra = -4.20, holgura = 0.00
grado_gasolina: precio sombra = -0.67, holgura = 0.00
grado_aceite: precio sombra = -0.65, holgura = 0.00
lim_gasolina: precio sombra = 4.48, holgura = 0.00
lim_aceite: precio sombra = 3.80, holgura = 0.00


---
## Ejercicio 4 - Donovan Enterprises (Minimización)

**Variables:**
- `r1, r2, r3, r4`: empleados que descansan en trimestre 1, 2, 3, 4
- `I1, I2, I3, I4`: inventario final de cada trimestre

**Demanda:** Q1=4000, Q2=2000, Q3=3000, Q4=10000. Inv. inicial=600

**Producción trimestre i** = 500 × (empleados que NO descansan en i)

**Objetivo:** Minimizar z = 30000(r1+r2+r3+r4) + 30(I1+I2+I3+I4)

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var r1 >= 0;  # empleados que descansan trimestre 1
var r2 >= 0;  # empleados que descansan trimestre 2
var r3 >= 0;  # empleados que descansan trimestre 3
var r4 >= 0;  # empleados que descansan trimestre 4
var I1 >= 0;  # inventario final trimestre 1
var I2 >= 0;  # inventario final trimestre 2
var I3 >= 0;  # inventario final trimestre 3
var I4 >= 0;  # inventario final trimestre 4

# definicion de funcion objetivo
# costo mano de obra + costo inventario

minimize costo: 30000 * (r1 + r2 + r3 + r4) + 30 * (I1 + I2 + I3 + I4);

# definicion de restricciones
# balance inventario: inv_prev + produccion - demanda = inv_final
# produccion trimestre i = 500 * (empleados que NO descansan en i)

s.t. trim1: 600 + 500 * (r2 + r3 + r4) - 4000  = I1;
s.t. trim2: I1  + 500 * (r1 + r3 + r4) - 2000  = I2;
s.t. trim3: I2  + 500 * (r1 + r2 + r4) - 3000  = I3;
s.t. trim4: I3  + 500 * (r1 + r2 + r3) - 10000 = I4;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 506000
4 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados planificacion de produccion:")

cant_r1 = ampl.get_variable('r1').value()
cant_r2 = ampl.get_variable('r2').value()
cant_r3 = ampl.get_variable('r3').value()
cant_r4 = ampl.get_variable('r4').value()
cant_I1 = ampl.get_variable('I1').value()
cant_I2 = ampl.get_variable('I2').value()
cant_I3 = ampl.get_variable('I3').value()
cant_I4 = ampl.get_variable('I4').value()
costo = ampl.get_objective('costo').value()

total_empleados = cant_r1 + cant_r2 + cant_r3 + cant_r4

print("total empleados: ", total_empleados)
print("descansan trimestre 1: ", cant_r1)
print("descansan trimestre 2: ", cant_r2)
print("descansan trimestre 3: ", cant_r3)
print("descansan trimestre 4: ", cant_r4)
print("inventario final Q1: ", cant_I1)
print("inventario final Q2: ", cant_I2)
print("inventario final Q3: ", cant_I3)
print("inventario final Q4: ", cant_I4)
print("costo minimo: ", costo)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados planificacion de produccion:
total empleados:  12.266666666666666
descansan trimestre 1:  5.466666666666666
descansan trimestre 2:  6.8
descansan trimestre 3:  0.0
descansan trimestre 4:  0.0
inventario final Q1:  0.0
inventario final Q2:  733.3333333333334
inventario final Q3:  3866.666666666667
inventario final Q4:  0.0
costo minimo:  506000.0

Analisis de sensibilidad
trim1: precio sombra = -10.00, holgura = 0.00
trim2: precio sombra = -10.00, holgura = -0.00
trim3: precio sombra = 20.00, holgura = -0.00
trim4: precio sombra = 50.00, holgura = 0.00


---
## Ejercicio 5 - Encuesta Telefónica (Minimización)

**Variables:**
- `xd`: llamadas de día (costo $2)
- `xn`: llamadas de noche (costo $5)

**Objetivo:** Minimizar z = 2xd + 5xn

Necesita detectar: 150 esposas, 120 esposos, 100 varones solteros, 110 mujeres solteras

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var xd >= 0;  # llamadas de dia
var xn >= 0;  # llamadas de noche

# definicion de funcion objetivo

minimize costo: 2 * xd + 5 * xn;

# definicion de restricciones

s.t. esposas: 0.30 * xd + 0.30 * xn >= 150;
s.t. esposos: 0.10 * xd + 0.30 * xn >= 120;
s.t. var_soltero: 0.10 * xd + 0.15 * xn >= 100;
s.t. muj_soltera: 0.10 * xd + 0.20 * xn >= 110;
s.t. max_nocturnas: xn <= xd;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2300
3 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados de la encuesta:")

cant_xd = ampl.get_variable('xd').value()
cant_xn = ampl.get_variable('xn').value()
costo   = ampl.get_objective('costo').value()

print("llamadas de dia: ", cant_xd)
print("llamadas de noche: ", cant_xn)
print("total llamadas: ", cant_xd + cant_xn)
print("costo minimo: ", costo)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados de la encuesta:
llamadas de dia:  900.0
llamadas de noche:  100.0
total llamadas:  1000.0
costo minimo:  2300.0

Analisis de sensibilidad
esposas: precio sombra = 0.00, holgura = 150.00
esposos: precio sombra = 10.00, holgura = 0.00
var_soltero: precio sombra = 0.00, holgura = 5.00
muj_soltera: precio sombra = 10.00, holgura = 0.00
max_nocturnas: precio sombra = 0.00, holgura = 800.00


---
## Ejercicio 6 - Brady Corporation (Minimización)

**Variables:**
- `x1`: pies³ tablones grado 1 (costo compra $3, rendimiento 0.7, secado 1.2s/pie)
- `x2`: pies³ tablones grado 2 (costo compra $7, rendimiento 0.9, secado 0.8s/pie)
- `x3`: pies³ troncos propios (corte $3 + aserradero $2.5, rendimiento 0.8, secado 1.3s/pie)

**Objetivo:** Minimizar z = compra + secado + aserradero + corte

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var x1 >= 0;  # pies cubicos tablones grado 1
var x2 >= 0;  # pies cubicos tablones grado 2
var x3 >= 0;  # pies cubicos troncos propios

# definicion de funcion objetivo
# compra + secado + aserradero + corte

minimize costo: (3 * x1 + 7 * x2) + (4 * x1 + 4 * x2 + 4 * x3) + 2.5 * x3 + 3 * x3;

# definicion de restricciones

s.t. demanda_min: 0.7 * x1 + 0.9 * x2 + 0.8 * x3 >= 90000;
s.t. cap_aserradero: x3 <= 35000;
s.t. lim_grado1: x1 <= 40000;
s.t. lim_grado2: x2 <= 60000;
s.t. tiempo_secado: 1.2 * x1 + 0.8 * x2 + 1.3 * x3 <= 144000;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 1028055.556
0 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados de abastecimiento de tablones:")

cant_x1 = ampl.get_variable('x1').value()
cant_x2 = ampl.get_variable('x2').value()
cant_x3 = ampl.get_variable('x3').value()
costo = ampl.get_objective('costo').value()

print("tablones grado 1 (pies3): ", cant_x1)
print("tablones grado 2 (pies3): ", cant_x2)
print("troncos propios (pies3): ", cant_x3)
print("madera util total: ", 0.7 * cant_x1 + 0.9 * cant_x2 + 0.8 * cant_x3)
print("costo minimo semanal: ", costo)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados de abastecimiento de tablones:
tablones grado 1 (pies3):  40000.0
tablones grado 2 (pies3):  37777.77777777777
troncos propios (pies3):  35000.0
madera util total:  90000.0
costo minimo semanal:  1028055.5555555555

Analisis de sensibilidad
demanda_min: precio sombra = 12.22, holgura = 0.00
cap_aserradero: precio sombra = -0.28, holgura = 0.00
lim_grado1: precio sombra = -1.56, holgura = 0.00
lim_grado2: precio sombra = 0.00, holgura = 22222.22
tiempo_secado: precio sombra = 0.00, holgura = 20277.78


---
## Ejercicio 7 - Centro de Reciclaje (Minimización)

**Variables:**
- `xA`: toneladas chatarra A ($100/ton) → 6% Al, 3% Si, 4% C
- `xB`: toneladas chatarra B ($80/ton) → 3% Al, 6% Si, 3% C

**Objetivo:** Minimizar z = 100xA + 80xB

Producir 1000 toneladas con: Al ∈ [3%,6%], Si ∈ [3%,5%], C ∈ [3%,7%]

In [ ]:
ampl = ampl_notebook(modules=["highs"], license_uuid="default")

model = '''

# definicion de variables

var xA >= 0;  # toneladas chatarra A
var xB >= 0;  # toneladas chatarra B

# definicion de funcion objetivo

minimize costo: 100 * xA + 80 * xB;

# definicion de restricciones

s.t. produccion: xA + xB = 1000;

s.t. aluminio_min: 0.06 * xA + 0.03 * xB >= 30;
s.t. aluminio_max: 0.06 * xA + 0.03 * xB <= 60;

s.t. silicio_min: 0.03 * xA + 0.06 * xB >= 30;
s.t. silicio_max: 0.03 * xA + 0.06 * xB <= 50;

s.t. carbon_min: 0.04 * xA + 0.03 * xB >= 30;
s.t. carbon_max: 0.04 * xA + 0.03 * xB <= 70;

'''


Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).


In [ ]:
ampl.eval(model)
ampl.option["solver"] = 'highs'
ampl.solve()

HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 86666.66667
0 simplex iterations
0 barrier iterations


In [ ]:
print("Resultados de la mezcla optima:")

cant_xA = ampl.get_variable('xA').value()
cant_xB = ampl.get_variable('xB').value()
costo = ampl.get_objective('costo').value()

aluminio = (0.06 * cant_xA + 0.03 * cant_xB) / 1000 * 100
silicio = (0.03 * cant_xA + 0.06 * cant_xB) / 1000 * 100
carbon = (0.04 * cant_xA + 0.03 * cant_xB) / 1000 * 100

print("chatarra A (ton): ", cant_xA)
print("chatarra B (ton): ", cant_xB)
print("aluminio %: ", round(aluminio, 2), " (rango: 3%-6%)")
print("silicio  %: ", round(silicio, 2), " (rango: 3%-5%)")
print("carbon   %: ", round(carbon, 2), " (rango: 3%-7%)")
print("costo minimo: ", costo)

print("\nAnalisis de sensibilidad")
for name, con in ampl.get_constraints():
    print(f"{name}: precio sombra = {con.dual():.2f}, holgura = {con.slack():.2f}")

Resultados de la mezcla optima:
chatarra A (ton):  333.3333333333334
chatarra B (ton):  666.6666666666666
aluminio %:  4.0  (rango: 3%-6%)
silicio  %:  5.0  (rango: 3%-5%)
carbon   %:  3.33  (rango: 3%-7%)
costo minimo:  86666.66666666667

Analisis de sensibilidad
produccion: precio sombra = 120.00, holgura = 0.00
aluminio_min: precio sombra = 0.00, holgura = 10.00
aluminio_max: precio sombra = 0.00, holgura = 20.00
silicio_min: precio sombra = 0.00, holgura = 20.00
silicio_max: precio sombra = -666.67, holgura = 0.00
carbon_min: precio sombra = 0.00, holgura = 3.33
carbon_max: precio sombra = 0.00, holgura = 36.67
